# Patrón Estructural: Proxy

## Introducción
El patrón Proxy proporciona un objeto sustituto o representante de otro objeto para controlar el acceso a este.

## Objetivos
- Comprender cómo controlar el acceso a objetos.
- Identificar cuándo es útil el patrón Proxy.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Acceso a archivos remotos**
Supón que tienes un sistema que accede a archivos remotos. El patrón Proxy permite controlar el acceso, por ejemplo, añadiendo caché o control de permisos.

**¿Dónde se usa en proyectos reales?**
En sistemas de archivos, proxies de red, control de acceso, etc.

## Sin patrón Proxy (forma errónea)
El cliente accede directamente al recurso, sin control ni intermediación.

In [1]:
class ArchivoRemoto:
    def leer(self):
        print('Leyendo archivo remoto...')

archivo = ArchivoRemoto()
archivo.leer()

Leyendo archivo remoto...


## Con patrón Proxy (forma correcta)
El cliente accede al recurso a través de un proxy que puede añadir control de acceso, caché, etc.

In [2]:
class ProxyArchivoRemoto:
    def __init__(self):
        self._archivo = ArchivoRemoto()
        self._cache = None
    def leer(self):
        if self._cache is None:
            print('Accediendo por primera vez, leyendo archivo...')
            self._archivo.leer()
            self._cache = 'datos'
        else:
            print('Usando caché')

proxy = ProxyArchivoRemoto()
proxy.leer()
proxy.leer()

Accediendo por primera vez, leyendo archivo...
Leyendo archivo remoto...
Usando caché


## UML del patrón Proxy
```plantuml
@startuml
class ArchivoRemoto {
    + leer()
}
class ProxyArchivoRemoto {
    + leer()
}
ProxyArchivoRemoto --> ArchivoRemoto
@enduml
```

## Otro ejemplo de la vida real: Proxy de protección (control de acceso)
**Contexto:** el ejemplo anterior usa un *proxy de caché*. Existe otro tipo muy común: el **proxy de protección**, que verifica permisos antes de dejar pasar la llamada al objeto real. Supón un servicio de administración de usuarios donde solo los administradores pueden eliminar cuentas — el servicio real no debería tener que preocuparse por quién lo está llamando.

### Sin patrón (forma errónea)
Cualquier código que tenga una referencia al servicio puede invocar la operación sensible, sin ninguna verificación de rol.

In [3]:
class ServicioUsuarios:
    def eliminar_usuario(self, usuario_id):
        print(f'Usuario {usuario_id} eliminado')

servicio = ServicioUsuarios()
servicio.eliminar_usuario(42)  # cualquier código puede llamarlo, sin validar el rol de quien llama

Usuario 42 eliminado


### Con patrón (forma correcta)
`ProxyAutorizacionUsuarios` intercepta la llamada, valida el rol, y solo delega al servicio real si el usuario tiene permiso — el `ServicioUsuarios` real no sabe nada de roles.

In [4]:
class ProxyAutorizacionUsuarios:
    def __init__(self, rol_actual):
        self._servicio = ServicioUsuarios()
        self._rol_actual = rol_actual
    def eliminar_usuario(self, usuario_id):
        if self._rol_actual != 'admin':
            print('Acceso denegado: se requiere rol admin')
            return
        self._servicio.eliminar_usuario(usuario_id)


proxy_editor = ProxyAutorizacionUsuarios(rol_actual='editor')
proxy_editor.eliminar_usuario(42)

proxy_admin = ProxyAutorizacionUsuarios(rol_actual='admin')
proxy_admin.eliminar_usuario(42)

Acceso denegado: se requiere rol admin
Usuario 42 eliminado


### UML del ejemplo de autorización
```plantuml
@startuml
class ServicioUsuarios {
    + eliminar_usuario(usuario_id)
}
class ProxyAutorizacionUsuarios {
    - _servicio: ServicioUsuarios
    - _rol_actual
    + eliminar_usuario(usuario_id)
}
ProxyAutorizacionUsuarios --> ServicioUsuarios
@enduml
```

### ¿Dónde más se usa Proxy?
- **Proxies de autorización:** exactamente este ejemplo — frameworks como Django REST Framework o Spring Security interceptan la petición antes de llegar al controlador real.
- **Proxy virtual (carga perezosa):** en una galería web, cada imagen se representa con un objeto liviano que solo descarga la imagen real cuando entra en el viewport (lazy loading).
- **Proxy remoto:** un cliente gRPC/REST generado automáticamente actúa como proxy de un servicio que en realidad vive en otro servidor.
- **Proxy de registro (logging):** un wrapper que registra cada llamada a un servicio (qué se llamó, con qué argumentos, cuánto tardó) antes de delegar al objeto real.
- **Proxy de caché:** el ejemplo con el que abre este notebook — evitar repetir una operación costosa si el resultado ya se calculó.

**Ejercicio de reflexión:** ¿podrías combinar en una sola clase el proxy de caché del primer ejemplo con el proxy de autorización de este? ¿En qué orden delegarías las validaciones (primero permisos, luego caché, o al revés) y por qué importa el orden?

## Actividad
Crea un proxy para controlar el acceso a una base de datos o a un servicio web, añadiendo control de permisos o caché.

---
## Explicación de conceptos clave
- **Control de acceso:** Permite añadir lógica antes o después de acceder al objeto real.
- **Transparencia:** El cliente usa el proxy como si fuera el objeto real.
- **Aplicación en la vida real:** Útil en sistemas de archivos, redes y control de acceso.

## Conclusión
El patrón Proxy es ideal para controlar el acceso a recursos y añadir funcionalidades como caché, control de permisos o registro de acceso.